In [1]:
%matplotlib inline
import sys
sys.path.append("/global/cfs/projectdirs/m4491/btaczak/SIBIL-runs/helper_scripts")

In [3]:
import os
import sys
import yaml
import numpy as np
from pathlib import Path
import re
import numpy as np
import netCDF4 as nc
import yaml
from scipy.interpolate import interp1d
from boutdata.data import BoutData
from boututils.datafile import DataFile
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FormatStrFormatter
from boutdata.data import BoutData
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, fixed
from sibil_io_functions import load_config, sibil_config_summary, confirm, parse_tdata2, parse_block_sizes, write_updated_namelist, _format_value, extract_tdata_first_index, interpolate_dataset, get_midplane_distances_from_psin, vec_summary, set_tdata2_block, get_bout_midplane_distances
from converged_profile_functions import unbiased_convergence, end_biased_convergence

DEFAULT_CONFIG_PATH = "SIBIL_config.yaml"

jomp = 44
profile_sample_timestep = -5 # Where in the convergence is the profile take from (-1 is last timestep)
                             # NOTE: profile_sample_timestep >= N for the model to make sense
N = 100
l = 196

config_path = DEFAULT_CONFIG_PATH

if not os.path.exists(config_path):
    print(f"Error: config file not found: {config_path}")
    sys.exit(1)

config = load_config(config_path)
sibil_config_summary(config)

# if not confirm():
#     print("Aborted. No changes made.")
#     sys.exit(0)

import sibil  # imported here so summary/confirmation works without dependency installed

# Matches any tdata(i, j, k, l) = value — tolerates arbitrary whitespace.
_TDATA_RE = re.compile(
    r"(tdata\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)\s*=\s*)"
    r"([+-]?\d+(?:\.\d*)?(?:[eE][+-]?\d+)?)",
    re.IGNORECASE,
)

# Matches ndata(1, k, l) = N  — used to discover block sizes and identifiers.
_NDATA_RE = re.compile(
    r"ndata\(\s*1\s*,\s*(\d+)\s*,\s*(\d+)\s*\)\s*=\s*(\d+)",
    re.IGNORECASE,
)

INPUT_FILE  = f"{config["base_path"]}/{config["iteration"]}_iteration/b2.transport.inputfile"
OUTPUT_FILE = f"{config["base_path"]}/{int(config["iteration"])+1}_iteration/b2.transport.inputfile"
bout_dir  = Path(f"{config["base_path"]}/{config["iteration"]}_iteration/Shelf_new_grid2_sinks2_gradfix")
data_dir  = bout_dir / "numpy"
grid_fname= bout_dir / "d3d_shelf_184016.03200_nc3.grd.nc"
grid_fname = str(grid_fname)
d = BoutData(path=bout_dir)

grid = DataFile(grid_fname)
try:
    _ = grid["psi_bdry"]
    has_psi_bdry = True
except Exception:
    has_psi_bdry = False
    
if has_psi_bdry:
    psi_bndry = grid["psi_bdry"]
else:
    psi_bndry = grid["psi_bndry"]

psi_midplane = grid["psixy"][:,jomp]
psin_midplane = ((grid["psixy"] - grid["psi_axis"]) / (psi_bndry - grid["psi_axis"]))[:,jomp]

sepx_dists_solps = extract_tdata_first_index(INPUT_FILE)
psin_mid, sepx_dists_bout = get_bout_midplane_distances(grid_fname)


transport_coeffs = ["Da_t","Xe_t","Xi_t"]
convergent_transport_coeffs = {}
interplted_transport_coeffs = {}

for var_name in transport_coeffs:
    var = np.load(f"{data_dir}/{var_name}.npy")
    var_conv = unbiased_convergence(var, N, l)
    var_conv_midplane = var_conv[profile_sample_timestep,:,jomp]
    var_conv_interpolated = interpolate_dataset(sepx_dists_bout,var_conv_midplane,sepx_dists_solps)
    nan_mask = np.isnan(var_conv_interpolated)
    var_conv_interpolated[nan_mask]=0
    
    convergent_transport_coeffs[var_name] = var_conv
    interplted_transport_coeffs[var_name] = var_conv_interpolated

SIBIL Midplane Mapper — Configuration Summary
  G-file         : ../g184016.03200  [OK]
  Geometry file  : ../baserun/HvsD_Dmesh_noC_v1.v001.geo  [OK]
  B-grid file    : Shelf_new_grid2_sinks2_gradfix/d3d_shelf_184016.03200_nc3.grd.nc  [OK]
  b2fstate       : b2fstate  [OK]
  b2fplasmf      : b2fplasmf  [MISSING]
  b2fgmtry       : b2fgmtry  [OK]

  Softmax scales (max, min):
    ni  : max=1e+16, min=1e+15
    ne  : max=1e+16, min=1e+15
    Ti  : max=0.5, min=0.0001
    Te  : max=0.5, min=0.0001


/global/homes/b/btaczak/.conda/envs/bout-data/lib/python3.13/site-packages/boutdata/data.py:765: AlwaysWarning: While building x, y, z coordinate arrays, an exception occured: [Errno 2] No such file or directory: '"d3d_shelf_184016.03200_nc3.grd.nc"'
Evaluating non-scalar options not available
  alwayswarn(


In [ ]:
# PLOTTING
# -----------------------------------------------------------------------------
# 3) PLOT FUNCTION -- creates a brand-new figure each call (inline-friendly)
# -----------------------------------------------------------------------------
colors_conv = {"D_conv": "C0", "Xe_conv": "C1", "Xi_conv": "C2"}
colors = {"D": "C0", "Xe": "C1", "Xi": "C2"}
labels = {"D": r"$D$", "Xe": r"$\chi_e$", "Xi": r"$\chi_i$"}

def plot_profiles(j, t=0):
    x = psin
    fig, axs = plt.subplots(3, 1, figsize=(4, 5), sharex=True)

    for ax, name in zip(axs, arrays.keys()):
        y = get_slice(arrays[name], j, t)
        ax.plot(x, y, color=colors[name], label=labels[name])
        ax.set_ylabel(labels[name])
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis="x", labelsize=7)
        ax.tick_params(axis="y", labelsize=7)
        ax.yaxis.get_offset_text().set_fontsize(8)

    axs[2].set_xlabel(r"$\psi_n$")
    title = f"Poloidal index = {j}"
    if has_time:
        title += f"   |   Time index = {t}"
    #fig.suptitle(title)
    # fig.tight_layout()
    plt.show()

# -----------------------------------------------------------------------------
# 4) WIRE UP SLIDERS -- run this cell once
# -----------------------------------------------------------------------------
if has_time:
    interact(
        plot_profiles,
        j=IntSlider(min=0, max=n_pol - 1, step=1, value=0, description="Pol. idx:"),
        #t=IntSlider(min=0, max=n_time - 1, step=1, value=0, description="Time idx:"),
        t=IntSlider(min=0, max=l - 1, step=1, value=0, description="Time idx:"),
    )
else:
    interact(
        plot_profiles,
        j=IntSlider(min=0, max=n_pol - 1, step=1, value=0, description="Pol. idx:"),
        t=fixed(0),
    )
    
for i, var in enumerate(transport_coeffs):
    transport_coeff = var.split("_")[0]
    ylabel = f"{transport_coeff[:-1]}_{transport_coeff[-1]}"
    axs[i].scatter(values,profile/np.linalg.norm(profile),label=f"{var}")
    axs[i].set_ylabel(f"${ylabel}$")
    axs[i].legend(fontsize="xx-small", handlelength=1.5)
    if i == rows-1:
        axs[i].set_xlabel(r"${\psi}_n$")

fig.tight_layout()

outpath = f"{config["base_path"]}/{config["iteration"]}_iteration/{config["to_bout_plot"]}"
fig.savefig(outpath, dpi=100)
print(f"Saved plot to {outpath}")
plt.close()

In [ ]:
a